In [1]:
import sys, torch
sys.path.insert(0, "qwen3_5")                      # match the architecture of the checkpoint
from model import TinyQwen35
from tokenizer import CharTokenizer


In [3]:

ckpt = torch.load("qwen3_5/tiny_qwen35.pt", map_location="cpu", weights_only=False)
tok = CharTokenizer(ckpt["chars"])
model = TinyQwen35(ckpt["cfg"]); model.load_state_dict(ckpt["model"]); model.eval()

model

TinyQwen35(
  (embed_tokens): Embedding(34, 32)
  (layers): ModuleList(
    (0-2): 3 x TransformerBlock(
      (input_layernorm): RMSNorm()
      (mixer): GatedDeltaNet(
        (q_proj): Linear(in_features=32, out_features=32, bias=False)
        (k_proj): Linear(in_features=32, out_features=32, bias=False)
        (v_proj): Linear(in_features=32, out_features=32, bias=False)
        (o_proj): Linear(in_features=32, out_features=32, bias=False)
        (alpha_proj): Linear(in_features=32, out_features=4, bias=True)
        (beta_proj): Linear(in_features=32, out_features=4, bias=True)
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=32, out_features=64, bias=False)
        (up_proj): Linear(in_features=32, out_features=64, bias=False)
        (down_proj): Linear(in_features=64, out_features=32, bias=False)
      )
    )
    (3): TransformerBlock(
      (input_layernorm): RMSNorm()
      (mixer): Attention(
        (q_proj): 

In [4]:
model.embed_tokens

Embedding(34, 32)

In [8]:
tok.encode("ABACI")

[5, 6, 5, 7, 13]

In [10]:
model.embed_tokens.weight

Parameter containing:
tensor([[ 0.1543, -0.1295, -0.3966,  ..., -0.4658,  0.9970,  1.9861],
        [-0.6121, -0.1486,  0.7727,  ...,  0.6409,  1.5571,  0.1542],
        [-0.6064, -0.2207,  0.2569,  ..., -0.1237,  0.0141, -0.4618],
        ...,
        [ 0.6072,  0.7895,  0.7257,  ...,  0.7370,  0.0606,  1.0619],
        [ 0.5257,  0.1517, -0.1923,  ...,  0.5910, -0.0874,  0.8755],
        [ 0.1155,  0.5876, -0.1419,  ..., -0.5770, -0.0633, -0.5099]],
       requires_grad=True)

In [11]:
start = torch.full((10, 1), tok.eos_id, dtype=torch.long)   # every name starts at EOS/newline
out = model.generate(start, max_new_tokens=model.cfg.max_seq_len,
                     temperature=0.8, eos_id=tok.eos_id)     # stops each row at EOS
out

tensor([[ 0, 26,  9, 18,  9, 12,  5, 18,  0,  0,  0,  0,  0,  0],
        [ 0, 15,  5, 27, 13, 28,  5, 26, 16,  5, 15,  0,  0,  0],
        [ 0, 11,  9,  8, 32, 15, 23, 19, 22, 24, 18,  0,  0,  0],
        [ 0, 15,  5, 21,  5, 15, 19, 26, 24, 18, 19, 16, 24,  0],
        [ 0, 11, 30, 21,  6, 24, 16, 11, 24, 21,  0,  0,  0,  0],
        [ 0,  6,  5, 26, 13, 21,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  5, 16, 22, 24, 27, 16, 24,  0,  0,  0,  0,  0,  0],
        [ 0,  9, 16, 15,  5, 26,  5,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  8,  5, 26,  5, 21,  5, 18, 16,  5, 21,  0,  0,  0],
        [ 0,  5, 15, 28,  5, 15, 19, 27,  0,  0,  0,  0,  0,  0]])

In [12]:
for row in out.tolist():
    print(tok.decode(row[1:]).split("\n")[0])

YENEHAN
KAZIÇAYLAK
GEDİKTOSUN
KARAKOYUNOLU
GÜRBULGUR
BAYIR
ALSUZLU
ELKAYA
DAYARANLAR
AKÇAKOZ
